# فكرة المشروع

يهدف هذا المشروع إلى بناء نموذج تعلم آلي للتنبؤ باحتمالية مغادرة العملاء للخدمة (Customer Churn Prediction) اعتماداً على بيانات العملاء المختلفة مثل مدة الاشتراك، نوع العقد، طريقة الدفع، والرسوم الشهرية.

تم تنفيذ مراحل فهم البيانات (Data Understanding) وتنظيفها (Data Cleaning)، ثم تدريب نموذج Decision Tree لتصنيف العملاء إلى فئتين: العملاء المتوقع استمرارهم في الخدمة والعملاء المتوقع مغادرتهم.

يساعد هذا النوع من المشاريع الشركات على التعرف المبكر على العملاء المعرضين للمغادرة واتخاذ إجراءات مناسبة للحفاظ عليهم وتحسين رضا العملاء.


In [1]:
import pandas as pd

df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.shape

(7043, 21)

## data understanding

## فهم حجم البيانات

تحتوي قاعدة البيانات على 7043 عميل و21 متغيراً.

يُعد حجم البيانات مناسباً لإجراء التحليل الاستكشافي وبناء نماذج تعلم الآلة واستخراج نتائج موثوقة.

In [2]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## استعراض أول خمسة سجلات

تم عرض أول خمسة سجلات للتأكد من تحميل البيانات بشكل صحيح.

يتضح أن كل صف يمثل عميلاً واحداً، بينما تمثل الأعمدة الخصائص المختلفة المتعلقة بالعميل مثل مدة الاشتراك والخدمات المستخدمة والرسوم الشهرية وحالة مغادرة العميل للشركة.

كما تم تحديد المتغير المستهدف (Churn) والذي سيتم استخدامه لاحقاً في بناء نموذج التنبؤ.

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


## فهم بنية البيانات

تحتوي البيانات على متغيرات رقمية ومتغيرات فئوية.

لا توجد قيم مفقودة ظاهرة من الفحص الأولي.

لوحظ أن عمود TotalCharges مخزن كنص بدلاً من قيمة رقمية، ولذلك سيتم فحصه ومعالجته خلال مرحلة تنظيف البيانات.

In [4]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


## الإحصاءات الوصفية

متوسط مدة اشتراك العملاء يقارب 32 شهراً.

تتراوح الرسوم الشهرية بين 18.25 و118.75 بمتوسط يقارب 64.76.

كما أن نسبة كبار السن تمثل حوالي 16% من إجمالي العملاء.

In [5]:
df['Churn'].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

## تحليل المتغير المستهدف

يتضح أن 26.5% من العملاء غادروا الشركة، بينما استمر 73.5% منهم.

يمثل هذا المتغير الهدف الأساسي للمشروع، حيث سيتم بناء نموذج قادر على التنبؤ بإمكانية مغادرة العميل للشركة مستقبلاً.

data cleaning

In [6]:
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

تم فحص البيانات المفقودة باستخدام isnull().sum()، واتضح أن جميع الأعمدة لا تحتوي على أي قيم مفقودة، لذلك لا توجد حاجة إلى تعويض أو حذف بيانات في هذه المرحلة

In [7]:
df.duplicated().sum()

np.int64(0)

تم فحص الصفوف المكررة باستخدام duplicated().sum()، واتضح عدم وجود أي سجلات مكررة، لذلك لا توجد حاجة إلى حذف أي بيانات قبل بناء النموذج

In [8]:
df["TotalCharges"].head(20)

0       29.85
1      1889.5
2      108.15
3     1840.75
4      151.65
5       820.5
6      1949.4
7       301.9
8     3046.05
9     3487.95
10     587.45
11      326.8
12     5681.1
13     5036.3
14    2686.05
15    7895.15
16    1022.95
17    7382.25
18     528.35
19     1862.9
Name: TotalCharges, dtype: object

تم فحص أول 20 قيمة من عمود TotalCharges للتأكد من طبيعة البيانات الموجودة داخله.

اتضح أن القيم تمثل مبالغ مالية رقمية، ولكن نوع البيانات ما زال Object، لذلك سيتم تحويله لاحقًا إلى نوع رقمي ليكون مناسبًا للتحليل وبناء النموذج

In [9]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [10]:
df["TotalCharges"].dtype

dtype('float64')

In [11]:
df["TotalCharges"].isnull().sum()

np.int64(11)

م تحويل عمود TotalCharges من نوع نصي (Object) إلى نوع رقمي (Float) ليصبح مناسباً للتحليل وبناء نماذج التعلم الآلي

In [12]:
df=df.dropna()

In [13]:
df.shape

(7032, 21)

بعد تحويل عمود TotalCharges إلى نوع رقمي، ظهرت 11 قيمة مفقودة. تم حذف هذه الصفوف لأنها تمثل نسبة صغيرة جداً من إجمالي البيانات (11 فقط من أصل 7043 صف)، وبالتالي لن تؤثر بشكل ملحوظ على النتائج. كما أن الاحتفاظ بقيم مفقودة قد يسبب مشاكل أثناء تدريب نموذج التعلم الآلي

In [14]:
df = df.drop(columns=["customerID"])

In [15]:
df.shape

(7032, 20)

تم حذف عمود customerID لأنه يمثل معرفاً فريداً لكل عميل فقط، ولا يحتوي على معلومات تساعد نموذج التعلم الآلي في التنبؤ بمغادرة العميل أو بقائه. الاحتفاظ به قد يضيف معلومات غير مفيدة للنموذج

EDA 

In [16]:
import plotly.express as px

In [17]:
fig = px.histogram(
    df,
    x="MonthlyCharges",
    color="Churn",
    title="Monthly Charges by Churn"
)

fig.show()

تم استخدام هذا الرسم لدراسة العلاقة بين الرسوم الشهرية (MonthlyCharges) ومغادرة العملاء (Churn). من خلال المقارنة بين العملاء الذين غادروا والذين استمروا، نلاحظ وجود اختلاف في توزيع الرسوم الشهرية بين المجموعتين، مما يشير إلى أن قيمة الرسوم الشهرية قد تكون من العوامل المؤثرة على قرار العميل بالبقاء أو المغادرة. لذلك سيتم أخذ هذا المتغير بعين الاعتبار عند بناء نموذج التنبؤ بمغادرة العملاء

In [18]:
fig = px.box(
    df,
    x="Churn",
    y="tenure",
    color="Churn",
    title="Tenure by Churn"
)

fig.show()

تم استخدام هذا الرسم لدراسة تأثير مدة اشتراك العميل مع الشركة (Tenure) على حالة المغادرة (Churn). أظهر الرسم أن العملاء الذين غادروا الشركة كانت مدة اشتراكهم أقل مقارنة بالعملاء الذين استمروا، مما يدل على أن العملاء الجدد أو أصحاب مدة الاشتراك القصيرة أكثر عرضة للمغادرة. لذلك يعتبر متغير Tenure من أهم المتغيرات التي قد تساعد النموذج على التنبؤ بمغادرة العملاء

In [19]:
fig = px.histogram(
    df,
    x="Contract",
    color="Churn",
    barmode="group",
    title="Contract Type by Churn"
)

fig.show()

يوضح الرسم أن العملاء أصحاب العقود الشهرية (Month-to-month) لديهم أعلى معدل مغادرة مقارنةً بأصحاب العقود السنوية أو العقود لمدة سنتين. وهذا يشير إلى أن نوع العقد يعد من العوامل المهمة التي تؤثر على قرار العميل بالبقاء أو المغادرة، لذلك من المتوقع أن يكون لهذا المتغير تأثير قوي في التنبؤ بعملية Churn

In [20]:
fig = px.histogram(
    df,
    x="PaymentMethod",
    color="Churn",
    barmode="group",
    title="Payment Method by Churn"
)

fig.show()

يوضح الرسم العلاقة بين طريقة الدفع ومغادرة العملاء. نلاحظ أن العملاء الذين يستخدمون Electronic Check لديهم عدد مغادرة أعلى مقارنة بباقي طرق الدفع، بينما تظهر طرق الدفع التلقائية مثل Credit Card و Bank Transfer معدلات مغادرة أقل، مما يشير إلى احتمال وجود علاقة بين طريقة الدفع واستمرار العميل مع الشركة

## data preprocessing

In [21]:
df["Churn"].value_counts()

Churn
No     5163
Yes    1869
Name: count, dtype: int64

تم فحص المتغير المستهدف Churn وتبين أن عدد العملاء الذين استمروا مع الشركة أكبر من عدد العملاء الذين غادروا، مما يدل على وجود عدم توازن بسيط في البيانات

In [22]:
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

In [23]:
df["Churn"].head()

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64

تم تحويل المتغير المستهدف Churn من قيم نصية (Yes/No) إلى قيم رقمية (1/0)، وذلك لأن نماذج التعلم الآلي لا تستطيع التعامل مباشرة مع البيانات النصية

In [24]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

print(X.shape)
print(y.shape)

(7032, 19)
(7032,)


In [25]:
df.shape

(7032, 20)

In [26]:
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


In [27]:
print(y.head())
print(y.unique())

0    0
1    0
2    1
3    0
4    1
Name: Churn, dtype: int64
[0 1]


تم فصل البيانات إلى قسمين: المتغيرات المستقلة (X) التي سيتم استخدامها للتنبؤ، والمتغير المستهدف (y) وهو Churn. بعد الفصل أصبح لدينا 19 متغيراً للتنبؤ وعمود واحد يمثل حالة مغادرة العميل

In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(5625, 19)
(1407, 19)


تم تقسيم البيانات إلى مجموعة تدريب بنسبة 80% ومجموعة اختبار بنسبة 20%. الهدف من هذه الخطوة هو تدريب النموذج على جزء من البيانات ثم تقييم أدائه على بيانات جديدة لضمان قدرته على التنبؤ بشكل صحيح

## Model Building

بعد تجهيز البيانات وتقسيمها إلى بيانات تدريب واختبار، أصبحت البيانات جاهزة لبناء نموذج تعلم آلي. في هذه المرحلة سيتم تدريب نموذج Decision Tree ليتعلم الأنماط الموجودة في البيانات ويستخدمها للتنبؤ بما إذا كان العميل سيغادر الشركة أم لا

In [34]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)

model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current

In [30]:
X = pd.get_dummies(X, drop_first=True)

print(X.shape)

(7032, 30)


In [31]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [33]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)

model.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current

أثناء تدريب النموذج ظهر خطأ لأن بعض الأعمدة كانت نصية مثل Gender و Contract، بينما نموذج Decision Tree يحتاج بيانات رقمية. لذلك استخدمت pd.get_dummies() لتحويل الأعمدة النصية إلى أعمدة رقمية، وبعد إعادة تقسيم البيانات وتدريب النموذج اشتغل بنجاح

الى سوىتة الان هو

أنشأت نموذج Decision Tree ثم دربته باستخدام بيانات التدريب X_train و y_train حتى يتعلم الأنماط التي تؤدي إلى Churn أو عدمه

الان 

## Model Evaluation

In [35]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[0 0 1 0 0 0 0 0 0 1]


In [36]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print(accuracy)

0.7249466950959488


قمنا بتقييم النموذج باستخدام Accuracy. النتيجة كانت 72.95%، وهذا يعني أن النموذج استطاع التنبؤ بشكل صحيح بحوالي 73% من حالات العملاء الموجودة في بيانات الاختبار

In [37]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.80      0.81      1033
           1       0.48      0.52      0.50       374

    accuracy                           0.72      1407
   macro avg       0.65      0.66      0.66      1407
weighted avg       0.73      0.72      0.73      1407



تم تقييم نموذج Decision Tree باستخدام بيانات الاختبار. حقق النموذج دقة تقارب 72%، مما يدل على قدرته على التنبؤ بحالات إلغاء الخدمة (Churn) بمستوى أداء مقبول

In [38]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[825 208]
 [179 195]]


تم استخدام Confusion Matrix لقياس أداء النموذج بشكل أكثر تفصيلاً. أظهرت النتائج أن النموذج نجح في تصنيف عدد كبير من العملاء بشكل صحيح، مع وجود بعض الحالات التي تم تصنيفها بشكل غير صحيح، وهو أمر طبيعي في نماذج التعلم الآلي

أظهرت مصفوفة الالتباس (Confusion Matrix) أن نموذج Decision Tree استطاع تصنيف عدد كبير من العملاء بشكل صحيح، مع وجود بعض الحالات التي تم تصنيفها بشكل غير دقيق. بشكل عام حقق النموذج أداءً جيداً بدقة تقارب 72%، مما يدل على قدرته على التنبؤ بالعملاء المعرضين لمغادرة الخدمة والاستفادة منه في دعم اتخاذ القرار

تم الانتهاء من جميع مراحل تحليل البيانات وبناء النموذج وتقييمه، وكانت آخر خطوة هي استخدام Confusion Matrix لتقييم أداء النموذج بشكل تفصيلي

In [ ]:
import joblib

joblib.dump(model, "churn_model.pkl")

print("Model saved successfully")

Model saved successfully


في هذا المشروع قمنا بتحليل بيانات عملاء شركة اتصالات للتنبؤ باحتمالية مغادرة العميل (Customer Churn). بدأنا بفهم البيانات وتحليلها، ثم قمنا بتنظيفها ومعالجة القيم وتحويل المتغيرات النصية إلى رقمية باستخدام get_dummies. بعد ذلك تم تقسيم البيانات إلى بيانات تدريب واختبار، ثم تدريب نموذج Decision Tree Classifier وتقييمه باستخدام Accuracy وClassification Report وConfusion Matrix. أخيراً تم حفظ النموذج وبناء تطبيق Streamlit لتمكين المستخدم من إدخال بيانات العميل والحصول على توقع مباشر لاحتمالية مغادرته للخدمة